# Constructing the flatbug example dataset


## Setup & helpers

In [1]:
import os
import requests
from remotezip import RemoteZip, RemoteIOError
from collections import defaultdict

def get_zip_download_url(record_id):
    """
    Fetches the Zenodo manifest and finds the download URL for the zip file.
    """
    api_url = f"https://zenodo.org/api/records/{record_id}"
    response = requests.get(api_url)
    response.raise_for_status()
    
    files = response.json().get('files', [])
    
    # Find the first file that is a .zip
    for file_info in files:
        if file_info['key'].endswith('.zip'):
            return file_info['links']['self']
            
    raise RuntimeError("No .zip file found in this record.")

def extract_subset_from_remote_zip(zip_url, extension_filter, download_dir="downloads"):
    """
    Connects to a remote zip and extracts only files matching the extension.
    """
    os.makedirs(download_dir, exist_ok=True)
    
    print("Connecting to remote zip (this may take a moment to read the directory)...")
    
    try:
        with RemoteZip(zip_url) as rz:
            # 1. List all files inside the remote zip
            all_files = rz.namelist()
            
            # 2. Filter for the ones you want
            target_files = [f for f in all_files if f.endswith(extension_filter)]
            print(f"Found {len(target_files)} files matching '{extension_filter}' inside the archive.\n")
            
            # 3. Extract only those files
            for file_name in target_files:
                print(f"Extracting {file_name}...")
                rz.extract(file_name, path=download_dir)
                
            print("\nDone! All targeted files extracted.")
            
    except RemoteIOError as e:
        print(f"Failed to read remote zip. The server might not support Range requests properly, or the URL is invalid: {e}")

def extract_complex_subset(
        zip_url, 
        max_directories=5,
        max_jpegs=20, 
        jsons_to_grab=1, 
        download_dir="downloads"
    ):
    """
    Connects to a remote zip, groups files by folder, and extracts a specific 
    quota of JPEGs and JSONs per folder.
    """
    os.makedirs(download_dir, exist_ok=True)
    print("Connecting to remote zip to read directory structure...")
    
    try:
        with RemoteZip(zip_url) as rz:
            all_files = rz.namelist()
            
            # This dictionary will group our files. 
            # Format: { 'folderA': {'jpegs': [...], 'jsons': [...]}, 'folderB': ... }
            folder_data = defaultdict(lambda: {'jpegs': [], 'jsons': []})
            
            # 1. Categorize all files by folder and type
            for file_path in all_files:
                # Skip directories themselves (in zips, they end with '/')
                if file_path.endswith('/'):
                    continue
                    
                folder_name = os.path.dirname(file_path)
                
                # Get the extension (lowercased to handle .JPEG, .jpg, etc.)
                ext = file_path.lower().split('.')[-1]
                
                if ext in ['jpeg', 'jpg']:
                    folder_data[folder_name]['jpegs'].append(file_path)
                elif ext == 'json':
                    folder_data[folder_name]['jsons'].append(file_path)
                    
            # 2. Apply your limits to build the final list of files to download
            files_to_extract = []
            
            for i, (folder, files) in zip(range(max_directories), folder_data.items()):
                # Take up to the maximum allowed JPEGs
                selected_jpegs = files['jpegs'][:max_jpegs]
                
                # Take the required number of JSONs 
                # (Change to `files['jsons']` without slicing if you want ALL jsons in the folder)
                selected_jsons = files['jsons'][:jsons_to_grab]
                
                files_to_extract.extend(selected_jpegs)
                files_to_extract.extend(selected_jsons)
                
            print(f"Found {len(folder_data)} folders.")
            print(f"Selected a total of {len(files_to_extract)} files for extraction.\n")
            
            if not files_to_extract:
                print("No matching files found based on your criteria.")
                return

            # 3. Extract the targeted subset
            for idx, file_path in enumerate(files_to_extract, 1):
                print(f"[{idx}/{len(files_to_extract)}] Extracting {file_path}...")
                rz.extract(file_path, path=download_dir)
                
            print("\nDone! All targeted files extracted.")
            
    except RemoteIOError as e:
        print(f"Failed to read remote zip. Error: {e}")

## Get the dataset zip URL

In [39]:
RECORD_ID = "14761447"
DOWNLOAD_DIR = os.path.join(".", "download")

zip_url = get_zip_download_url(RECORD_ID)
print(zip_url)

https://zenodo.org/api/records/14761447/files/flatbug-dataset.zip/content


## Download the subset

In [ ]:
extract_complex_subset(zip_url, download_dir=DOWNLOAD_DIR)

Connecting to remote zip to read directory structure...
Found 23 folders.
Selected a total of 105 files for extraction.

[1/105] Extracting flatbug-dataset/ALUS/Araneae_Unknown_2020_10_16_4334.jpg...
[2/105] Extracting flatbug-dataset/ALUS/Araneae_Unknown_2020_10_23_4410.jpg...
[3/105] Extracting flatbug-dataset/ALUS/Araneae_Unknown_2020_11_09_4719.jpg...
[4/105] Extracting flatbug-dataset/ALUS/Araneae_Unknown_2020_11_09_4720.jpg...
[5/105] Extracting flatbug-dataset/ALUS/Araneae_Unknown_2020_11_09_4721.jpg...
[6/105] Extracting flatbug-dataset/ALUS/Araneae_Unknown_2020_11_10_4722.jpg...
[7/105] Extracting flatbug-dataset/ALUS/Araneae_Unknown_2020_11_10_4723.jpg...
[8/105] Extracting flatbug-dataset/ALUS/Araneae_Unknown_2020_11_10_4724.jpg...
[9/105] Extracting flatbug-dataset/ALUS/Araneae_Unknown_2020_11_10_4725.jpg...
[10/105] Extracting flatbug-dataset/ALUS/Araneae_Unknown_2020_11_10_4726.jpg...
[11/105] Extracting flatbug-dataset/ALUS/Araneae_Unknown_2020_11_10_4727.jpg...
[12/105]

## Postprocess

We need to filter the labels found in the instances_default.json files to ensure they only contain labels for images which we actually include in the example dataset.

In [40]:
import json
from glob import glob

example_data_files = glob(os.path.join(DOWNLOAD_DIR, "*/*/*"))
example_subdatasets = {}
for file in example_data_files:
    subdataset = os.path.basename(os.path.dirname(file))
    if subdataset not in example_subdatasets:
        example_subdatasets[subdataset] = {
            "label_file" : None,
            "images" : []
        }
    if file.lower().endswith("jpeg") or file.lower().endswith("jpg") or file.lower().endswith("png"):
        example_subdatasets[subdataset]["images"].append(file)
        if len(example_subdatasets[subdataset]["images"]) > 20:
            raise RuntimeError(f'{subdataset=} contains too many images {len(example_subdatasets[subdataset]["images"])} > 20:\n' + "\n".join(example_subdatasets[subdataset]["images"]))
    if file.lower().endswith("json"):
        if example_subdatasets[subdataset]["label_file"] is not None:
            raise RuntimeError(f'{subdataset=} already contains labels: {example_subdatasets[subdataset]["label_file"]}, but received {file}')
        example_subdatasets[subdataset]["label_file"] = file
        with open(file) as f:
            example_subdatasets[subdataset]["labels"] = json.load(f)

COPY_KEYS = ("licenses", "info", "categories")
FILTER_KEYS= ("images", "annotations")
for subdataset, data in example_subdatasets.items():
    orig_labels = data["labels"]    

    filtered_images = [os.path.basename(file) for file in data["images"]]
    filtered_ids = [img["id"] for img in orig_labels["images"] if img["file_name"] in filtered_images]

    filtered_labels = {}
    for k in COPY_KEYS:
        filtered_labels[k] = orig_labels[k]
    for k in FILTER_KEYS:
        filtered_labels[k] = [e for e in orig_labels[k] if e["id"] in filtered_ids]
    
    data["filtered_labels"] = filtered_labels

## Create the example dataset

In [48]:
import shutil

EXAMPLE_DIR = os.path.join("..", "data")

for subdataset, data in example_subdatasets.items():
    os.makedirs(os.path.join(EXAMPLE_DIR, subdataset), exist_ok=True)
    for image in data["images"]:
        shutil.copy2(image, os.path.join(EXAMPLE_DIR, os.path.relpath(image, os.path.join(DOWNLOAD_DIR, "flatbug-dataset"))))
    with open(os.path.join(EXAMPLE_DIR, subdataset, "instances_default.json"), "w") as f:
        json.dump(data["filtered_labels"], f)